# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and examine its general information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Authors (by @id): {[a['@id'] for a in getattr(metadata, 'author', [])]}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")
print(f"Published on: {getattr(metadata, 'datePublished', '')}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The dataset contains structured tables for clinical features, hormone levels, and genetic variants as described in metadata.
We will retrieve the available `RecordSet` objects and their corresponding fields.

In [ ]:
# List all RecordSets and their IDs
record_sets = list(dataset.record_sets())

if not record_sets:
    print("No RecordSets found directly in metadata. Attempting auto-discovery.")
    # Try to discover record sets from the metadata's tables, fields, or by convention

    # In FAIR^2 Croissant datasets, tables are typically named and have their own @id.
    # mlcroissant auto-discovers and sometimes exposes them as e.g. 'Table 1', 'Table 2', 'Table 3'.

    # Use dataset.record_sets() to get their IDs
    record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
    if not record_sets:
        # Fallback logic, sometimes mlcroissant exposes them as auto-names
        record_sets = [rs for rs in dataset.list_record_sets()] # e.g. 'Table 1', ...

print("RecordSets IDs found:")
for rs_id in record_sets:
    print(f"- {rs_id}")

# Show the fields (columns) for each RecordSet
for rs_id in record_sets:
    # List all fields @id for this record set
    fields = dataset.fields(record_set=rs_id)
    print(f"\nRecordSet {rs_id}: Fields (@id)")
    for field in fields:
        print(f"  - Field @id: {field['@id']}    Name: {field.get('name', '')}")

## 3. Data Extraction
Load the data from all available RecordSets into pandas DataFrames using their `@id`.

We will demonstrate extracting data from each RecordSet.

In [ ]:
# Load all records from each RecordSet into a DataFrame
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"RecordSet: {rs_id} | Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"RecordSet {rs_id} contains no records.")

## 4. Exploratory Data Analysis (EDA)
This section demonstrates filtering, normalizing, and grouping data by key attributes.

We will select a numeric field (for example, 'height' or 'HormoneLevel' depending on available columns), filter patients above a threshold, normalize values, and group by a categorical field where available.

In [ ]:
# Example EDA: Pick a RecordSet and numeric field

# Choose the first RecordSet that has records
eda_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        eda_rs_id = rs_id
        break

if eda_rs_id:
    df = dataframes[eda_rs_id]
    # Try to select a numeric field from columns
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']] or [col for col in df.columns if 'height' in col.lower() or 'age' in col.lower()]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Try grouping by a categorical field
        possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print(f"No numeric field found in RecordSet {eda_rs_id}, columns are: {df.columns.tolist()}")
else:
    print("No RecordSet with non-empty records found.")

## 5. Visualization
Visualize the distribution of a numeric variable in one of the record sets.

For example, if 'height' or 'age' is present, we'll plot a histogram.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if eda_rs_id and numeric_fields:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {eda_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using the mlcroissant library, explored its record sets and fields (referenced by `@id`), extracted tabular data into DataFrames, and performed basic EDA and visualization. The dataset provides valuable insights into clinical and genetic features, but its small size and focused cohort offer limited generalizability. Further studies with expanded data are recommended for robust analysis and predictive modeling.